In [9]:
import importlib
import sys
from pathlib import Path
import pandas as pd

# Set up home path and working repo dir 
HOME = Path.home()
repo_dir = HOME / "OneDrive/git-repos/LCBP-interannual-EMMAs"

if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

# Import EMMA module
import EMMA.event_emma as em  
import EMMA.event_pca_error as ep
import EMMA.uncertainty_propagation as up
importlib.reload(ep)
importlib.reload(em)
importlib.reload(up)

# Define data directory using relative path logic
data_dir = repo_dir / "Data/GrabSample_data"

# Define output directory using relative path logic
output_dir = repo_dir / "Output/EMMA-uncertainty"

# Load the full RI25 dataset
df = pd.read_csv(data_dir / "RI23-IC-ICP-isotope-toc-joined.csv")

####################
# Wade RI23 events #
####################

wade_tracers = ['Ca_mg_L', 'Si_mg_L', 'Mg_mg_L', 'dD', 'd18O', 'Na_mg_L']

# 1. Run EMMA function
(
    wade_febros_fractions_df,
    wade_febros_scaler,
    wade_febros_pca,
    wade_febros_endmembers_df,
) = em.run_emma_event(
    data=df,
    site="Wade",
    start_date="2023-02-14 00:00:00",
    end_date="2023-02-20 00:00:00",
    endmember_ids=["RI23-5006", "RI23-5018", "RI23-5000", "RI23-5005"],
)

# 2. Extract raw endmember rows matching these IDs (to calculate endmember SDs)
em_raw_subset = df[df["Sample ID"].isin(["RI23-5006", "RI23-5018", "RI23-5000", "RI23-5005"])]

# 3. Define streamwater samples for the event window
stream_event_df = df[
    (df["Site"] == "Wade")
    & (df["Type"].isin(["Grab", "Grab/Isco", "Baseflow", "Isco"]))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) >= pd.to_datetime("2023-02-14 00:00:00"))
    & (pd.to_datetime(df["Date"] + " " + df["Time"]) <= pd.to_datetime("2023-02-20 00:00:00"))
].copy()
stream_event_df["Datetime"] = pd.to_datetime(
    stream_event_df["Date"] + " " + stream_event_df["Time"]
)

# 4. Define tracer analytical uncertainties (Optional: edit these to match lab detection limits)
analytical_sd = {
    "Ca_mg_L": 0.05,  # mg/L
    "Si_mg_L": 0.10,  # mg/L
    "Mg_mg_L": 0.02,  # mg/L
    "dD": 0.8,  # per mil (stable isotope precision)
    "d18O": 0.08,  # per mil
    "Na_mg_L": 0.05,  # mg/L
}

# 5. Calculate Genereux Uncertainties
uncertainty_df = up.propagate_genereux_uncertainty(
    stream_df=stream_event_df,
    em_grouped=wade_febros_endmembers_df,
    em_raw=em_raw_subset,
    tracers=wade_tracers,
    analytical_sd=analytical_sd,
)

# 6. Merge fractions and their calculated uncertainties for a complete table
results_with_error = pd.merge(
    wade_febros_fractions_df, uncertainty_df, on=["Sample ID", "Datetime"]
)

# Identify which uncertainty columns were actually generated
uncertainty_cols = [
    col for col in results_with_error.columns if "Uncertainty_1sig" in col
]
# Extract the base fraction names (e.g., "Groundwater", "Snowmelt lysimeter")
fraction_cols = [col.replace("_Uncertainty_1sig", "") for col in uncertainty_cols]

# Combine them in alternating order: [Fraction_1, Uncertainty_1, Fraction_2, Uncertainty_2...]
display_cols = ["Sample ID", "Datetime"]
for frac, unc in zip(fraction_cols, uncertainty_cols):
    display_cols.extend([frac, unc])

# Print the head of the dynamically built column list
print(results_with_error[display_cols].head())
results_with_error.head(20)

   Sample ID            Datetime  Groundwater  Groundwater_Uncertainty_1sig  \
0  RI23-1009 2023-02-15 15:00:00     0.577292                      0.154078   
1  RI23-1010 2023-02-15 19:00:00     0.531795                      0.161962   
2  RI23-1011 2023-02-15 23:00:00     0.495768                      0.175489   
3  RI23-1025 2023-02-15 12:00:00     0.524279                      0.153148   
4  RI23-1012 2023-02-16 03:00:00     0.326576                      0.205359   

   Snowmelt lysimeter  Snowmelt lysimeter_Uncertainty_1sig  \
0       -6.342797e-14                         2.267527e-07   
1        3.879134e-13                         2.934114e-10   
2       -2.555990e-14                         9.031143e-10   
3       -4.660879e-13                         5.755614e-07   
4       -5.429684e-16                         3.129808e-09   

   Soil water lysimeter  Soil water lysimeter_Uncertainty_1sig  
0              0.422708                               0.154078  
1              0.46820

,Sample ID,Datetime,Site,Groundwater,Snowmelt lysimeter,Soil water lysimeter,Sum_Fractions,Groundwater_Uncertainty_1sig,Snowmelt lysimeter_Uncertainty_1sig,Soil water lysimeter_Uncertainty_1sig
0,RI23-1009,2023-02-15 15:00:00,Wade,0.577292,-6.342797e-14,0.422708,1.0,0.154078,2.267527e-07,0.154078
1,RI23-1010,2023-02-15 19:00:00,Wade,0.531795,3.879134e-13,0.468205,1.0,0.161962,2.934114e-10,0.161962
2,RI23-1011,2023-02-15 23:00:00,Wade,0.495768,-2.555990e-14,0.504232,1.0,0.175489,9.031143e-10,0.175489
3,RI23-1025,2023-02-15 12:00:00,Wade,0.524279,-4.660879e-13,0.475721,1.0,0.153148,5.755614e-07,0.153147
4,RI23-1012,2023-02-16 03:00:00,Wade,0.326576,-5.429684e-16,0.673424,1.0,0.205359,3.129808e-09,0.205359
5,RI23-1014,2023-02-16 11:00:00,Wade,0.229147,1.949860e-14,0.770853,1.0,0.222406,2.689215e-09,0.222406
6,RI23-1028,2023-02-16 14:00:00,Wade,0.231564,9.723918e-14,0.768436,1.0,0.217682,3.511234e-09,0.217682
7,RI23-1029,2023-02-16 20:00:00,Wade,0.292599,-1.250834e-15,0.707401,1.0,0.217076,2.875739e-09,0.217076
8,RI23-1030,2023-02-17 02:00:00,Wade,0.258380,-6.175271e-14,0.741620,1.0,0.208098,6.532470e-10,0.208098
9,RI23-1031,2023-02-17 08:00:00,Wade,0.323434,9.032271e-15,0.676566,1.0,0.201190,1.091492e-09,0.201190
